# AgentLab × Colab 单机一条龙(出片 + 人物 LoRA 训练)

**跑前准备(一次):**
1. 运行时改 A100 GPU(代码执行程序 → 更改运行时类型 → A100)。
2. 右侧🔑 Colab Secrets 加 `HF_TOKEN`(FLUX.1-dev 是 gated，需先在 HF 同意协议)。
3. 下面第 1 格填 `REPO_URL`(你的 AgentLab 仓库)和 `DEEPSEEK_KEY`。

**逐格点运行即可。每格幂等**(已装/已下/已起则跳过)，断连重跑不重下(模型软链到 Drive)。

顺序:部署 ComfyUI+后端 → 下模型 → 起服务 → 灌 EP1《ASHBORN》→ cloudflared 暴露 → (可选)训练人物 LoRA。

In [ ]:
# === 0. 参数 + GPU 自检 + 挂 Drive(已挂则跳过，避免 Mountpoint 报错)===
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'
BRANCH   = 'main'
DEEPSEEK_KEY = ''   # ← 填你的 DeepSeek key
import os
!nvidia-smi -L
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/agentlab_models', exist_ok=True)
os.makedirs('/content/cael_video_out', exist_ok=True)
print('Drive OK')

In [ ]:
# === 1. HuggingFace 登录(FLUX.1-dev 是 gated，必先登录)===
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get('HF_TOKEN')); print('HF 登录成功(Secret)')
except Exception:
    import getpass; login(getpass.getpass('粘贴 HF token(输入框不显示内容): ')); print('HF 登录成功(getpass)')

In [ ]:
# === 2. 拉 AgentLab 仓库(含模板/脚本/.env.colab/种子)，按分支克隆 ===
%cd /content
if not os.path.isdir('agentlab'):
    !git clone -b {BRANCH} {REPO_URL} agentlab
else:
    !cd agentlab && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/agentlab
!ls comfyui_workflows colab scripts/seed_ashborn_ep1.py

In [ ]:
# === 3. 装 ComfyUI + 自定义节点(幂等)===
!bash colab/setup.sh

In [ ]:
# === 4. 模型目录软链到 Drive(持久化，重连不重下;必须在装好 ComfyUI 之后)===
import os, shutil
CACHE = '/content/drive/MyDrive/agentlab_models'
for d in ['unet','clip','vae','audio_encoders','loras','pulid']:
    os.makedirs(f'{CACHE}/{d}', exist_ok=True)
    tgt = f'/content/ComfyUI/models/{d}'
    if os.path.islink(tgt):
        continue
    if os.path.isdir(tgt):
        shutil.rmtree(tgt)
    os.symlink(f'{CACHE}/{d}', tgt)
print('models 已软链到 Drive')

In [ ]:
# === 5. 下模型(出图 FLUX + i2v A14B 双专家 + s2v；幂等+扁平化)。首次约 80GB，走 Drive 缓存 ===
!bash colab/download_models.sh

In [ ]:
# === 6. 起 ComfyUI(127.0.0.1:8188，后台)===
import subprocess, time, urllib.request
subprocess.Popen(['python','/content/ComfyUI/main.py','--listen','127.0.0.1','--port','8188'],
                 stdout=open('/content/comfyui.log','w'), stderr=subprocess.STDOUT)
for _ in range(80):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3); print('ComfyUI 已就绪'); break
    except Exception: time.sleep(3)
else:
    print('ComfyUI 未就绪，看 /content/comfyui.log'); print(open('/content/comfyui.log').read()[-2000:])

In [ ]:
# === 7. 装后端 + 写干净 .env(总是覆盖，避免旧 .env 残留 → 出图回退 SSH)===
%cd /content/agentlab
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
import re, shutil
shutil.copy('.env.colab', '.env')          # 每次用干净模板覆盖
env = open('.env').read()
if DEEPSEEK_KEY:
    env = re.sub(r'OPENAI_API_KEY=.*', 'OPENAI_API_KEY=' + DEEPSEEK_KEY, env)
open('.env','w').write(env)
print('已写干净 .env(COMFYUI_IMAGE_AS=flux, i2v=A14B双专家, SSH 关)')

In [ ]:
# === 8. (可选)构建前端 → StaticFiles 单端口出 UI。慢/不需要可跳过，直接用 API ===
%cd /content/agentlab/frontend
!npm install --silent && npm run build || echo '前端构建跳过(用 API 即可)'
%cd /content/agentlab

In [ ]:
# === 9. 起后端(0.0.0.0:8000，后台)===
import subprocess, time, urllib.request
subprocess.Popen(['uvicorn','mirage.main_api:app','--host','0.0.0.0','--port','8000'],
                 cwd='/content/agentlab', stdout=open('/content/api.log','w'), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=3); print('后端已就绪'); break
    except Exception: time.sleep(2)
else:
    print('后端未就绪，看 /content/api.log'); print(open('/content/api.log').read()[-2000:])

In [ ]:
# === 10. 灌入《ASHBORN》EP1(项目+4角色+8分镜，对口型 2/5/8)===
!python scripts/seed_ashborn_ep1.py /content/cael_video_out

In [ ]:
# === 11. cloudflared 暴露后端 8000(拿公网地址打开 UI)===
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, time
subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000'],
                 stdout=open('/content/cf.log','w'), stderr=subprocess.STDOUT)
time.sleep(10)
print(open('/content/cf.log').read()[-1800:])
print('\n↑ 找 https://xxxx.trycloudflare.com 打开。工作目录选 /content/cael_video_out → 选中 ASHBORN → 一键全部出图 → 选图 → 一键出片并合成。')

In [ ]:
# === 12. 全自动出整集:出图 → 每镜自动选第1张 → 出片(A14B双专家)→ 合成。可重跑续做 ===
import requests, time
API='http://127.0.0.1:8000/api'; WS='/content/cael_video_out'
proj=lambda pid: requests.get(f"{API}/pipeline/project/{pid}",params={"workspace":WS},timeout=90).json()
pid=requests.get(f"{API}/pipeline/projects",params={"workspace":WS}).json()["projects"][0]["project_id"]
print("项目:", pid)
requests.post(f"{API}/pipeline/batch_generate",json={"project_id":pid,"workspace":WS},timeout=30)
t0=time.time()
while True:
    scs=proj(pid).get("scenes",[]); have=sum(1 for s in scs if s["candidates"])
    print(f"  出图 {have}/{len(scs)} ({int(time.time()-t0)}s)")
    if scs and have>=len(scs): break
    if time.time()-t0>3000: print("出图超时,看 comfyui.log"); break
    time.sleep(20)
for s in proj(pid)["scenes"]:
    if s["candidates"] and not s["selected"]:
        requests.post(f"{API}/pipeline/select",json={"scene_id":s["scene_id"],"asset_id":s["candidates"][0]["assetId"],"workspace":WS},timeout=30)
print("  已选图,开始出片+合成(A14B 双专家,慢,耐心)…")
requests.post(f"{API}/pipeline/batch_finish",json={"project_id":pid,"workspace":WS},timeout=30)
t0=time.time()
while True:
    try: p=proj(pid)
    except Exception as e: print("  查询失败,重试:",e); time.sleep(15); continue
    scs=p.get("scenes",[]); done=sum(1 for s in scs if s.get("video"))
    print(f"  出片 {done}/{len(scs)} 整集:{'✅' if p.get('episode') else '合成中...'} ({int(time.time()-t0)}s)")
    if p.get("episode"): print("🎬 整集成片:", p["episode"]["url"]); break
    if time.time()-t0>6000: print("出片超时,看 comfyui.log"); break
    time.sleep(30)

---
## 人物 LoRA 训练(角色一致性命脉)

虚构角色无照片 → 用 **PuLID 自举数据集**(从一张定妆脸固定身份批量出多样图)→ **ai-toolkit** 训 FLUX LoRA → 落 `ComfyUI/models/loras/` → 登记到项目 `flux_lora`/`trigger_word`,出图自动经 t2i 的 LoraLoader 接入。

**先训通 1 个角色(Caelan)**:改下面 `CHAR` 跑 L1-L8。Roland/Seraphina 改 `CHAR` 重跑。

> 数据集自举(种子脸 + PuLID 批量)那两步细节较多,先跑训练闭环;种子脸可先用『一键出图』给 Caelan 出几张挑 1 张当种子。

In [ ]:
# === L1. 装 ai-toolkit(训练器)+ PuLID 权重 ===
import os
CHAR='caelan'; TRIGGER=CHAR
AITK='/content/ai-toolkit'; WORK='/content/lora_work'; LORA_OUT='/content/ComfyUI/models/loras'
DS=f'{WORK}/{CHAR}_dataset'
for p in (WORK, DS, LORA_OUT): os.makedirs(p, exist_ok=True)
if not os.path.isdir(AITK):
    !git clone https://github.com/ostris/ai-toolkit {AITK}
    !cd {AITK} && git submodule update --init --recursive
!cd {AITK} && pip -q install -r requirements.txt
PW='/content/ComfyUI/models/pulid/pulid_flux_v0.9.1.safetensors'
if not (os.path.exists(PW) and os.path.getsize(PW)>0):
    !wget -q -O {PW} https://huggingface.co/guozinan/PuLID/resolve/main/pulid_flux_v0.9.1.safetensors
print('ai-toolkit + PuLID 就绪; CHAR=', CHAR)

In [ ]:
# === L2. 数据集(16-20 张)+ 打标 ===
# 把该角色的训练图放进 DS 目录(.png)。来源二选一:
#  (a) 用工作台『一键出图』给 Caelan 出多张,下载放进 DS;
#  (b) PuLID 自举:固定 1 张种子脸,用 pulid_t2i_template.json 循环换角度/表情批量出(进阶)。
import os
BASE={'caelan':'blonde male, sharp features, defiant',
      'roland':'blonde male noble, arrogant',
      'seraphina':'silver-hair noblewoman, haughty'}.get(CHAR,'character')
pngs=[f for f in os.listdir(DS) if f.lower().endswith(('.png','.jpg','.jpeg'))]
for f in pngs:
    open(os.path.join(DS, os.path.splitext(f)[0]+'.txt'),'w',encoding='utf-8').write(
        f'{TRIGGER}, {BASE}, portrait, detailed face')
print(f'数据集 {len(pngs)} 张,已打标。建议 16-20 张多角度/表情。')

In [ ]:
# === L3. 写训练配置(A100-40G:rank32 / 512 / bf16 / 2000步,约 45-90 分钟)===
import yaml, os
cfg={'job':'extension','config':{'name':f'{CHAR}_flux_lora','process':[{
  'type':'sd_trainer','training_folder':f'{WORK}/output','device':'cuda:0',
  'network':{'type':'lora','linear':32,'linear_alpha':16},
  'save':{'dtype':'float16','save_every':500,'max_step_saves_to_keep':2},
  'datasets':[{'folder_path':DS,'caption_ext':'txt','resolution':[512],'cache_latents_to_disk':True}],
  'train':{'batch_size':2,'steps':2000,'gradient_accumulation_steps':1,'train_unet':True,
           'train_text_encoder':False,'optimizer':'adamw8bit','lr':1e-4,'lr_scheduler':'cosine',
           'dtype':'bf16','gradient_checkpointing':True},
  'model':{'name_or_path':'black-forest-labs/FLUX.1-dev','is_flux':True,'quantize':False},
  'sample':{'sample_every':500,'prompts':[f'{TRIGGER}, portrait, studio lighting']},
}]}}
os.makedirs(f'{WORK}/output', exist_ok=True)
CFG=f'{WORK}/{CHAR}_train.yaml'; yaml.safe_dump(cfg, open(CFG,'w'))
print('训练配置:', CFG)

In [ ]:
# === L4. 开训 + 产物落 loras + 登记到项目(出图即用)===
import os, glob, shutil
FINAL=f'{LORA_OUT}/{CHAR}_lora.safetensors'
if os.path.exists(FINAL):
    print('已训过,跳过:', FINAL)
else:
    !cd {AITK} && python run.py {CFG}
    src=sorted(glob.glob(f'{WORK}/output/{CHAR}_flux_lora/*.safetensors'))[-1]
    shutil.copy(src, FINAL); print('LoRA →', FINAL)
# 登记到工作目录配置:出图自动注入 trigger + lora(无需重训即可用)
import sys; sys.path.insert(0,'/content/agentlab')
from mirage.app.pipeline import runtime
runtime.set_workspace('/content/cael_video_out')
runtime.set_model_config(trigger_word=TRIGGER, flux_lora=f'{CHAR}_lora.safetensors')
print('已登记 trigger=%s lora=%s_lora.safetensors → 出图自动加载' % (TRIGGER, CHAR))